CAPA GOLD

Creación de la stablas con los calculos agregados para análisis y visuaización interactiva.

In [ ]:
!pip install -q pyspark
import os, re, glob
from functools import reduce
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window
spark = (SparkSession.builder
         .master("local[*]")
         .appName("bronze")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .getOrCreate())

In [ ]:
# Conectar a la carpeta del drive
from google.colab import drive

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/nyc-taxi"

Mounted at /content/drive


In [ ]:
# cargar datos de capa silver
gold = spark.read.parquet(f"{BASE}/silver")
gold.printSchema()

root
 |-- dolocationid: integer (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- trip_minutes: double (nullable = true)
 |-- is_adjustment: boolean (nullable = true)
 |-- fare_unreliable: boolean (nullable = true)
 |-- passenger_unknown: boolean (nullable = true)
 |-- is_flex_fare: boolean (nullable = true)
 |-- duration_valid: boolean (nullable = true)
 |-- distance_valid: boolean (nul

# Creación de tablas

Listado de todas la stablas que se crean para el análisis

In [ ]:
# Tabla que calcula viajes por zona de recogida, hora y día de la semana
demanda = (gold.filter(~F.col("is_adjustment"))
                 .withColumn("hora", F.hour("pickup_datetime"))
                 .withColumn("dia_semana", F.dayofweek("pickup_datetime"))
                 .groupBy("taxi_type", "pulocationid", "pickup_borough", "pickup_zone",
                          "dia_semana", "hora")
                 .agg(F.count("*").alias("viajes")))

demanda.show(5)

+---------+------------+--------------+--------------------+----------+----+------+
|taxi_type|pulocationid|pickup_borough|         pickup_zone|dia_semana|hora|viajes|
+---------+------------+--------------+--------------------+----------+----+------+
|   yellow|          55|      Brooklyn|        Coney Island|         5|  12|    22|
|   yellow|         211|     Manhattan|                SoHo|         4|   1|    84|
|   yellow|         230|     Manhattan|Times Sq/Theatre ...|         1|  12|   853|
|   yellow|         114|     Manhattan|Greenwich Village...|         1|  14|   266|
|   yellow|          52|      Brooklyn|         Cobble Hill|         4|   2|     4|
+---------+------------+--------------+--------------------+----------+----+------+
only showing top 5 rows


In [ ]:
#Guardar tabla demanda
demanda.write.mode("overwrite").parquet(f"{BASE}/gold/demanda_zona_hora_dia")

In [ ]:
# Tabla que calcula el ingreso por minuto y zona
ingreso = (
    gold
    .filter(~F.col("is_adjustment") & F.col("duration_valid"))#quedarse solo con los viajes válidos
    .groupBy("taxi_type", "pulocationid", "pickup_borough", "pickup_zone")#agrupar por zona de recogida (y por tipo de taxi)
    .agg(F.count("*").alias("viajes"),
         F.round(F.sum(F.col("total_amount") - F.coalesce(F.col("tip_amount"), F.lit(0))),2).alias("ingreso_sin_propina"),
         F.round(F.sum("trip_minutes"),2).alias("minutos"))#en cada zona, sumar viajes, dinero y minutos
    .withColumn("ingreso_por_minuto", F.round(F.col("ingreso_sin_propina") / F.col("minutos"), 2))#dividir dinero entre minutos
    .filter(F.col("viajes") >= 50)#descartar zonas con muy pocos viajes (poco fiables)
)

ingreso.orderBy(F.desc("ingreso_por_minuto")).show(10)

+---------+------------+--------------+--------------------+------+-------------------+---------+------------------+
|taxi_type|pulocationid|pickup_borough|         pickup_zone|viajes|ingreso_sin_propina|  minutos|ingreso_por_minuto|
+---------+------------+--------------+--------------------+------+-------------------+---------+------------------+
|   yellow|           1|           EWR|      Newark Airport|   366|           32391.41|   830.17|             39.02|
|   yellow|         265|           N/A|      Outside of NYC|  1294|          120440.97|   9916.5|             12.15|
|   yellow|          93|        Queens|Flushing Meadows-...|   518|           36339.21|  9573.38|               3.8|
|    grenn|         264|       Unknown|                 N/A|    97|            1855.45|    580.7|               3.2|
|    grenn|         216|        Queens|    South Ozone Park|    58|            1565.25|    590.7|              2.65|
|   yellow|         138|        Queens|   LaGuardia Airport| 872

In [ ]:
# guardar tabla ingresos por minuto y zona
ingreso.write.mode("overwrite").parquet(f"{BASE}/gold/ingreso_minuto_zona")

In [ ]:
# Flujo de trayectos
recogidas = (gold.filter(~F.col("is_adjustment"))
                   .groupBy("taxi_type",
                            F.col("pulocationid").alias("zona_id"),
                            F.col("pickup_borough").alias("distrito_pu"),
                            F.col("pickup_zone").alias("zona_pu"),
                            F.hour("pickup_datetime").alias("hora"))
                   .agg(F.count("*").alias("recogidas")))

destinos = (gold.filter(~F.col("is_adjustment"))
                  .groupBy("taxi_type",
                           F.col("dolocationid").alias("zona_id"),
                           F.col("dropoff_borough").alias("distrito_do"),
                           F.col("dropoff_zone").alias("zona_do"),
                           F.hour("dropoff_datetime").alias("hora"))
                  .agg(F.count("*").alias("destinos")))

flujo_neto = (recogidas.join(destinos, on=["taxi_type", "zona_id", "hora"], how="full_outer")
                       .fillna(0, subset=["recogidas", "destinos"])
                       .withColumn("distrito", F.coalesce("distrito_pu", "distrito_do"))
                       .withColumn("zona", F.coalesce("zona_pu", "zona_do"))
                       .drop("distrito_pu", "distrito_do", "zona_pu", "zona_do")
                       .withColumn("flujo_neto", F.col("recogidas") - F.col("destinos")))

flujo_neto.show(5)

+---------+-------+----+---------+--------+--------+--------------+----------+
|taxi_type|zona_id|hora|recogidas|destinos|distrito|          zona|flujo_neto|
+---------+-------+----+---------+--------+--------+--------------+----------+
|    grenn|      1|   4|        0|       1|     EWR|Newark Airport|        -1|
|    grenn|      1|   7|        0|       1|     EWR|Newark Airport|        -1|
|    grenn|      1|   8|        0|       1|     EWR|Newark Airport|        -1|
|    grenn|      1|   9|        0|       1|     EWR|Newark Airport|        -1|
|    grenn|      1|  10|        0|       3|     EWR|Newark Airport|        -3|
+---------+-------+----+---------+--------+--------+--------------+----------+
only showing top 5 rows


In [ ]:
# Guardar flujo trayectos
flujo_neto.write.mode("overwrite").parquet(f"{BASE}/gold/flujo_neto_zona_hora")

In [ ]:
# Cantidad de propina pro zona de recogida
propina = (gold.filter(~F.col("is_adjustment") & ~F.col("fare_unreliable") &
                         ~F.col("is_flex_fare") & (F.col("payment_type") == 1) &
                         (F.col("fare_amount") > 0))
                 .groupBy("taxi_type", "pulocationid", "pickup_borough", "pickup_zone")
                 .agg(F.count("*").alias("viajes"),
                      F.sum(F.coalesce(F.col("tip_amount"), F.lit(0))).alias("propina_total"),
                      F.sum("fare_amount").alias("tarifa_total"))
                 .withColumn("propina_pct", F.round(100 * F.col("propina_total") / F.col("tarifa_total"), 2))
                 .filter(F.col("viajes") >= F.when(F.col("taxi_type") == "yellow", 100).otherwise(20)))

propina.show(5)

+---------+------------+--------------+--------------------+------+------------------+------------------+-----------+
|taxi_type|pulocationid|pickup_borough|         pickup_zone|viajes|     propina_total|      tarifa_total|propina_pct|
+---------+------------+--------------+--------------------+------+------------------+------------------+-----------+
|   yellow|         155|      Brooklyn|Marine Park/Mill ...|   300|137.51000000000002|          10269.51|       1.34|
|   yellow|          40|      Brooklyn|     Carroll Gardens|   184| 702.3299999999999|           4035.37|       17.4|
|   yellow|         140|     Manhattan|     Lenox Hill East| 54432| 184395.1700000062| 785620.9899999979|      23.47|
|   yellow|         125|     Manhattan|           Hudson Sq| 11615|43047.330000000096|178272.38999999964|      24.15|
|   yellow|         196|        Queens|           Rego Park|   154|            632.08|           7237.98|       8.73|
+---------+------------+--------------+-----------------

In [ ]:
# Guardar propina por zona
propina.write.mode("overwrite").parquet(f"{BASE}/gold/propina_zona")

In [ ]:
# Tipo de pago por zonas
pago_zona = (gold.filter(~F.col("is_adjustment"))
                   .groupBy("taxi_type", "pulocationid", "pickup_borough", "pickup_zone")
                   .agg(F.count("*").alias("viajes"),
                        F.count(F.when(F.col("payment_type") == 1, 1)).alias("tarjeta"),
                        F.count(F.when(F.col("payment_type") == 2, 1)).alias("efectivo"),
                        F.count(F.when(F.col("payment_type") == 0, 1)).alias("flex"),
                        F.count(F.when(F.col("payment_type").isNull() | ~F.col("payment_type").isin(0, 1, 2), 1)).alias("otros"))
                   .filter(F.col("viajes") >= F.when(F.col("taxi_type") == "yellow", 100).otherwise(20))
                   .withColumn("pct_tarjeta", F.round(100 * F.col("tarjeta") / F.col("viajes"), 2))
                   .withColumn("pct_efectivo", F.round(100 * F.col("efectivo") / F.col("viajes"), 2))
                   .withColumn("pct_flex", F.round(100 * F.col("flex") / F.col("viajes"), 2))
                   .withColumn("pct_otros", F.round(100 * F.col("otros") / F.col("viajes"), 2)))

pago_zona.show(5)

+---------+------------+--------------+--------------------+------+-------+--------+----+-----+-----------+------------+--------+---------+
|taxi_type|pulocationid|pickup_borough|         pickup_zone|viajes|tarjeta|efectivo|flex|otros|pct_tarjeta|pct_efectivo|pct_flex|pct_otros|
+---------+------------+--------------+--------------------+------+-------+--------+----+-----+-----------+------------+--------+---------+
|   yellow|         155|      Brooklyn|Marine Park/Mill ...|   579|    300|      13| 258|    8|      51.81|        2.25|   44.56|     1.38|
|   yellow|         200|         Bronx|Riverdale/North R...|   156|     77|       4|  73|    2|      49.36|        2.56|   46.79|     1.28|
|   yellow|          40|      Brooklyn|     Carroll Gardens|   803|    184|      35| 577|    7|      22.91|        4.36|   71.86|     0.87|
|   yellow|         201|        Queens|       Rockaway Park|   133|     68|       2|  63|    0|      51.13|         1.5|   47.37|      0.0|
|   yellow|         

In [ ]:
# Guardar tabla tipo de pago por zona
pago_zona.write.mode("overwrite").parquet(f"{BASE}/gold/pago_zona")

In [ ]:
# Tablas de flujo entre origen-destino

ventana_rutas = Window.partitionBy("taxi_type").orderBy(F.desc("viajes"))

rutas = (gold.filter(~F.col("is_adjustment"))
               .groupBy("taxi_type", "pulocationid", "pickup_zone", "dolocationid", "dropoff_zone")
               .agg(F.count("*").alias("viajes"),
                    F.round(F.sum("total_amount"), 2).alias("ingreso_total"))
               .filter(F.col("viajes") >= F.when(F.col("taxi_type") == "yellow", 50).otherwise(10))
               .withColumn("ranking", F.row_number().over(ventana_rutas)))

rutas.orderBy("taxi_type", "ranking").show(10)

+---------+------------+-----------------+------------+--------------------+------+-------------+-------+
|taxi_type|pulocationid|      pickup_zone|dolocationid|        dropoff_zone|viajes|ingreso_total|ranking|
+---------+------------+-----------------+------------+--------------------+------+-------------+-------+
|    grenn|          74|East Harlem North|          75|   East Harlem South|  2177|     30634.43|      1|
|    grenn|          75|East Harlem South|          74|   East Harlem North|  1439|     18889.73|      2|
|    grenn|          74|East Harlem North|         236|Upper East Side N...|  1309|      27524.5|      3|
|    grenn|          74|East Harlem North|         166| Morningside Heights|   788|     13406.87|      4|
|    grenn|          95|     Forest Hills|          95|        Forest Hills|   763|     10440.64|      5|
|    grenn|          75|East Harlem South|         236|Upper East Side N...|   733|     11725.46|      6|
|    grenn|          74|East Harlem North|    

In [ ]:
# Guardar flujo de rutas
rutas.write.mode("overwrite").parquet(f"{BASE}/gold/rutas_origen_destino")

In [ ]:
# Viajes al aeropuerto
ventana_tipo = Window.partitionBy("taxi_type")

largos = (gold.filter(~F.col("is_adjustment") & F.col("distance_valid"))
                .withColumn("tramo",
                            F.when(F.col("trip_distance") < 2, "0-2 millas")
                             .when(F.col("trip_distance") < 5, "2-5 millas")
                             .when(F.col("trip_distance") < 10, "5-10 millas")
                             .when(F.col("trip_distance") < 20, "10-20 millas")
                             .otherwise("20+ millas"))
                .groupBy("taxi_type", "is_airport", "tramo")
                .agg(F.count("*").alias("viajes"),
                     F.round(F.avg("total_amount"), 2).alias("tarifa_media"),
                     F.round(F.avg("trip_distance"), 2).alias("distancia_media"))
                .withColumn("pct_viajes", F.round(100 * F.col("viajes") / F.sum("viajes").over(ventana_tipo), 2)))

largos.show(20)

+---------+----------+------------+-------+------------+---------------+----------+
|taxi_type|is_airport|       tramo| viajes|tarifa_media|distancia_media|pct_viajes|
+---------+----------+------------+-------+------------+---------------+----------+
|    grenn|      true|  2-5 millas|    586|       28.41|           3.63|      1.29|
|    grenn|     false|  0-2 millas|  24752|       14.72|           1.22|      54.4|
|    grenn|      true|  20+ millas|     50|      117.54|          24.77|      0.11|
|    grenn|     false|  2-5 millas|  14737|       25.21|           2.95|     32.39|
|    grenn|     false|10-20 millas|    819|       66.97|          13.01|       1.8|
|    grenn|      true| 5-10 millas|    498|        46.3|           7.04|      1.09|
|    grenn|      true|  0-2 millas|     14|       21.65|            0.8|      0.03|
|    grenn|     false|  20+ millas|     94|      113.81|           28.0|      0.21|
|    grenn|      true|10-20 millas|    264|       79.08|          14.47|    

In [ ]:
# Guardar viajes al aeropuerto
largos.write.mode("overwrite").parquet(f"{BASE}/gold/aeropuertos_viajes_largos")

In [ ]:
# Donde compite los verdes contra los amarillos

# Cambiar grenn por green (en un futuro)
cuota_verde = (gold.filter(~F.col("is_adjustment"))
                     .groupBy("pulocationid", "pickup_borough", "pickup_zone")
                     .agg(F.count("*").alias("viajes_total"),
                          F.count(F.when(F.col("taxi_type") == "grenn", 1)).alias("viajes_verdes"),
                          F.count(F.when(F.col("taxi_type") == "yellow", 1)).alias("viajes_amarillos"))
                     .filter(F.col("viajes_total") >= 100)
                     .withColumn("pct_verde", F.round(100 * F.col("viajes_verdes") / F.col("viajes_total"), 2)))

cuota_verde.orderBy(F.desc("pct_verde")).show(10)

+------------+--------------+--------------------+------------+-------------+----------------+---------+
|pulocationid|pickup_borough|         pickup_zone|viajes_total|viajes_verdes|viajes_amarillos|pct_verde|
+------------+--------------+--------------------+------------+-------------+----------------+---------+
|          95|        Queens|        Forest Hills|        3632|         2235|            1397|    61.54|
|          74|     Manhattan|   East Harlem North|       20367|        11651|            8716|    57.21|
|          82|        Queens|            Elmhurst|        3256|         1789|            1467|    54.94|
|          97|      Brooklyn|         Fort Greene|        4470|         1991|            2479|    44.54|
|         134|        Queens|         Kew Gardens|        1240|          523|             717|    42.18|
|          83|        Queens|    Elmhurst/Maspeth|         867|          353|             514|    40.72|
|          92|        Queens|            Flushing|     

In [ ]:
# Guardar cuotas tipo verde
cuota_verde.write.mode("overwrite").parquet(f"{BASE}/gold/cuota_verde_zona")

In [ ]:
# Como consiguen lso taxis verdes a los pasajeros

# Cambiar grenn por green (en un futuro)
ventana_zona = Window.partitionBy("pulocationid")

tipo_viaje = (gold.filter((F.col("taxi_type") == "grenn") & ~F.col("is_adjustment"))
                    .withColumn("tipo_viaje",
                                F.when(F.col("trip_type") == 1, "calle")
                                 .when(F.col("trip_type") == 2, "despacho")
                                 .otherwise("desconocido"))
                    .groupBy("pulocationid", "pickup_borough", "pickup_zone", "tipo_viaje")
                    .agg(F.count("*").alias("viajes"))
                    .withColumn("pct_zona", F.round(100 * F.col("viajes") / F.sum("viajes").over(ventana_zona), 2)))

tipo_viaje.show(10)

+------------+--------------+--------------------+-----------+------+--------+
|pulocationid|pickup_borough|         pickup_zone| tipo_viaje|viajes|pct_zona|
+------------+--------------+--------------------+-----------+------+--------+
|           3|         Bronx|Allerton/Pelham G...|      calle|     2|   100.0|
|           6| Staten Island|Arrochar/Fort Wad...|desconocido|     1|   100.0|
|           7|        Queens|             Astoria|desconocido|   110|   12.78|
|           7|        Queens|             Astoria|      calle|   723|   83.97|
|           7|        Queens|             Astoria|   despacho|    28|    3.25|
|           8|        Queens|        Astoria Park|      calle|     1|   100.0|
|           9|        Queens|          Auburndale|   despacho|     1|    50.0|
|           9|        Queens|          Auburndale|      calle|     1|    50.0|
|          10|        Queens|        Baisley Park|desconocido|     3|   13.64|
|          10|        Queens|        Baisley Park|  

In [ ]:
# Guardar tabla tipo pasajero verdes
tipo_viaje.write.mode("overwrite").parquet(f"{BASE}/gold/tipo_viaje_verdes")

Tiempo de ejecución

Cargar datos de silver --> 2s.

Creación de tablas --> entre 0-8s.

Guardar las tablas --> al instante .